# Лабораторная работа № 2

Выполнил: студент гр. 6403-010302D Андрей Баландин

In [2]:
from pyspark.sql import SparkSession, functions as F

1. Найти велосипед с максимальным временем пробега.
2. Найти наибольшее геодезическое расстояние между станциями.
3. Найти путь велосипеда с максимальным временем пробега через станции.
4. Найти количество велосипедов в системе.
5. Найти пользователей потративших на поездки более 3 часов.

In [18]:
spark = SparkSession.builder.appName("Lab2_Balandin").getOrCreate()

trips = spark.read.csv("trips.csv", header=True, inferSchema=True)
stations = spark.read.csv("stations.csv", header=True, inferSchema=True)

print("Trips schema:")
trips.printSchema()
print("Stations schema:")
stations.printSchema()

Trips schema:
root
 |-- id: integer (nullable = true)
 |-- duration: integer (nullable = true)
 |-- start_date: string (nullable = true)
 |-- start_station_name: string (nullable = true)
 |-- start_station_id: integer (nullable = true)
 |-- end_date: string (nullable = true)
 |-- end_station_name: string (nullable = true)
 |-- end_station_id: integer (nullable = true)
 |-- bike_id: integer (nullable = true)
 |-- subscription_type: string (nullable = true)
 |-- zip_code: string (nullable = true)

Stations schema:
root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- long: double (nullable = true)
 |-- dock_count: integer (nullable = true)
 |-- city: string (nullable = true)
 |-- installation_date: string (nullable = true)



### Задание 1: Найти велосипед с максимальным временем пробега.

In [22]:
bike_time_stats = (trips.groupBy("bike_id").agg(F.sum("duration").alias("ride_time_sec")).sort(F.col("ride_time_sec").desc(), F.col("bike_id").asc()))
leader_bike = bike_time_stats.collect()[0]
bike_id_with_max_time = leader_bike["bike_id"]
total_time_sec = leader_bike["ride_time_sec"]
total_time_hours = total_time_sec / 3600.0
print(f"ID велосипеда с наибольшим временем поездок: {bike_id_with_max_time}")
print(f"Общее время поездок: {total_time_sec} сек.")
print(f"Общее время поездок в часах: {total_time_hours:.2f}")

ID велосипеда с наибольшим временем поездок: 593
Общее время поездок: 1217803 сек.
Общее время поездок в часах: 338.28


### Задание 2: Найти наибольшее геодезическое расстояние между станциями.

In [19]:
from pyspark.sql.functions import col, radians, sin, cos, atan2, sqrt

station_pairs = stations.alias("s1").join(stations.alias("s2"), col("s1.id") < col("s2.id"))

max_dist = station_pairs.withColumn("dlat", radians(col("s2.lat") - col("s1.lat"))) \
    .withColumn("dlon", radians(col("s2.long") - col("s1.long"))) \
    .withColumn("a", sin(col("dlat") / 2)**2 + cos(radians(col("s1.lat"))) * cos(radians(col("s2.lat"))) * sin(col("dlon") / 2)**2) \
    .withColumn("dist", 6371 * 2 * atan2(sqrt(col("a")), sqrt(1 - col("a")))) \
    .select("s1.name", "s2.name", "dist") \
    .orderBy(col("dist").desc()) \
    .limit(1)

max_dist.show(truncate=False)

+--------------------------+----------------------+----------------+
|name                      |name                  |dist            |
+--------------------------+----------------------+----------------+
|SJSU - San Salvador at 9th|Embarcadero at Sansome|69.9208759542826|
+--------------------------+----------------------+----------------+



### Задание 3: Найти путь велосипеда с максимальным временем пробега через станции.

In [20]:
from pyspark.sql.functions import to_timestamp

target_bike = bike_time_stats.collect()[0]['bike_id']

bike_path = trips.filter(col("bike_id") == target_bike) \
    .withColumn("start_date_dt", to_timestamp(col("start_date"), 'M/d/yyyy H:mm')) \
    .orderBy("start_date_dt") \
    .select("bike_id", "start_station_name", "end_station_name", "start_date")

bike_path.show(bike_path.count(), truncate=False)

+-------+---------------------------------------------+---------------------------------------------+----------------+
|bike_id|start_station_name                           |end_station_name                             |start_date      |
+-------+---------------------------------------------+---------------------------------------------+----------------+
|410    |Beale at Market                              |Commercial at Montgomery                     |8/29/2013 12:23 |
|410    |Commercial at Montgomery                     |Market at 10th                               |8/29/2013 13:16 |
|410    |Market at 10th                               |Powell Street BART                           |8/29/2013 17:09 |
|410    |Powell Street BART                           |Powell Street BART                           |8/30/2013 18:07 |
|410    |Powell Street BART                           |Washington at Kearney                        |9/2/2013 18:25  |
|410    |Washington at Kearney                  

### Задание 4: Найти количество велосипедов в системе.

In [23]:
bike_count = trips.select("bike_id").distinct().count()
print(f"Количество велосипедов в системе: {bike_count}")

Количество велосипедов в системе: 699


### Задание 5: Найти пользователей потративших на поездки более 3 часов.

In [24]:
users_3h_plus = trips.groupBy("zip_code").agg(F.sum("duration").alias("total_duration")).filter(col("total_duration") > 10800).select("zip_code", "total_duration")
users_3h_plus.show(10)

+--------+--------------+
|zip_code|total_duration|
+--------+--------------+
|   94102|       8260616|
|   95134|        373736|
|   84606|         42122|
|   80305|        148110|
|   60070|         28919|
|   95519|         12294|
|   91910|         42108|
|   48063|         12463|
|   95138|        115242|
|   94610|       1420086|
+--------+--------------+
only showing top 10 rows
